# LiDAR Point Cloud Visualization with PyBoreas and Plotly

This notebook demonstrates how to load a segment from pyboreas and visualize the LiDAR point cloud using plotly.

In [1]:
# Import necessary libraries
import plotly.graph_objects as go
import numpy as np
from pyboreas import BoreasDataset

root = "/data/boreas"
bd = BoreasDataset(root)
seq = bd.sequences[0]

In [2]:
def rotate_points_around_z(points, angle):
    """
    Rotate points around the z-axis by a given angle.

    Parameters:
    -----------
    points : numpy.ndarray of shape (N, 3)
        Array of points with x, y, z coordinates
    angle : float
        Rotation angle in radians (positive = counter-clockwise)

    Returns:
    --------
    rotated_points : numpy.ndarray of shape (N, 3)
        Rotated points
    """
    # Rotation matrix for rotation around z-axis
    cos_theta = np.cos(angle)
    sin_theta = np.sin(angle)

    # Rotation matrix
    R = np.array([
        [cos_theta, -sin_theta, 0], 
        [sin_theta, cos_theta, 0], 
        [0, 0, 1]
        ])

    # Apply rotation to each point
    rotated_points = points @ R

    return rotated_points

In [3]:
START_INDEX = 5432
indices = [START_INDEX + i for i in range(5)]

sample_points = []
for idx in indices:
    lidar_frame = seq.get_lidar(idx)
    sample_points.append(lidar_frame.points[:, :3])
    lidar_frame.unload_data()  # Memory reqs will keep increasing without this

In [4]:
from math import pi

MAX_DIST = 40.0

filtered_points = []
for points in sample_points:
    points_rot = rotate_points_around_z(points, pi / 4)
    norms = np.linalg.norm(points_rot, axis=1)  # Compute L2 norm for each point
    filtered_points.append(points_rot[norms < MAX_DIST])

In [5]:
def draw_lidar_cloud(input_points: np.ndarray, n_points: int = 10000):
    points = input_points[
        np.random.choice(input_points.shape[0], n_points, replace=False)
    ]

    # Create a 3D scatter plot for the LiDAR point cloud
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode="markers",
                marker=dict(
                    size=2,
                    color=points[:, 2],  # Color points by height
                    colorscale="Viridis",
                    opacity=0.7,
                ),
                name="LiDAR Points",
            )
        ]
    )

    # Update layout with custom styling
    fig.update_layout(
        title=dict(
            text="LiDAR Point Cloud Visualization", font=dict(size=20, family="Arial")
        ),
        scene=dict(
            xaxis_title="X (m)",
            yaxis_title="Y (m)",
            zaxis_title="Z (m)",
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode="data",
        ),
        height=900,
        width=1200,
    )
    return fig

## Visualize one frame

In [6]:
fig = draw_lidar_cloud(filtered_points[0])
# Show the plot
fig.show()